# Day 15 — The Full RAG Evaluation Report

> **Module 3 · RAG Evaluation**

So far, we evaluated different parts of RAG separately.

### Retriever

- Contextual Precision
- Contextual Recall
- Contextual Relevancy

### Generator

- Faithfulness
- Answer Relevancy

Today we combine all five metrics into one evaluation suite.

> **Core idea:** A RAG system does not have one "RAG score."  
> We diagnose which part of the pipeline is failing.

## 15.1 — Setup

We will use OpenAI for both:

- the RAG generator
- the DeepEval judge

We also limit DeepEval to **2 concurrent evaluation tasks**.

In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI
from deepeval.models import OpenAIModel
from deepeval.evaluate import AsyncConfig

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found."

client = OpenAI()

judge = OpenAIModel(
    model="gpt-4.1-mini",
    temperature=0,
)

async_config = AsyncConfig(
    max_concurrent=2
)

print("Judge:", judge.get_model_name())
print("Max concurrent:", async_config.max_concurrent)

Judge: gpt-4.1-mini
Max concurrent: 2


## 15.2 — Rebuild the RAG Pipeline

We use the same deliberately simple RAG application from Day 11.

```text
Question
   ↓
Retriever
   ↓
Retrieved Context
   ↓
LLM Generator
   ↓
Actual Answer

In [2]:
CORPUS = [
    {
        "title": "Tokens",
        "content": (
            "Large language models split text into tokens. "
            "Tokens are the basic units processed by an LLM, and both "
            "input and output are measured in tokens."
        ),
    },
    {
        "title": "Embeddings",
        "content": (
            "An embedding is a vector representation of text. "
            "Texts with similar meanings have vectors that are close together, "
            "which enables semantic search."
        ),
    },
    {
        "title": "Retrieval-Augmented Generation",
        "content": (
            "RAG grounds an LLM's answer in external documents. "
            "The system retrieves relevant information and provides it to "
            "the LLM as context, which can reduce hallucinations."
        ),
    },
    {
        "title": "Hallucination",
        "content": (
            "A hallucination is information generated by an LLM that is "
            "factually incorrect or unsupported by its sources."
        ),
    },
    {
        "title": "AI Agents",
        "content": (
            "An AI agent is an LLM given a goal, tools, and a loop that allows "
            "it to plan, take actions, observe results, and continue until "
            "the task is complete."
        ),
    },
]


def retrieve(query: str, k: int = 2) -> list[str]:
    """Simple keyword-overlap retriever."""

    query_words = set(query.lower().split())
    scored_docs = []

    for doc in CORPUS:
        text = f"{doc['title']} {doc['content']}".lower()
        doc_words = set(text.split())

        score = len(query_words & doc_words)
        scored_docs.append((score, doc))

    scored_docs.sort(
        key=lambda item: item[0],
        reverse=True
    )

    return [
        doc["content"]
        for _, doc in scored_docs[:k]
    ]

## 15.3 — Add the Generator

The generator must answer using only the retrieved context.

If the information is unavailable, it should refuse rather than invent an answer.

In [3]:
SYSTEM_PROMPT = """
Answer the question using only the provided context.

If the context does not contain enough information,
say that you do not have enough information.

Be concise.
""".strip()


def generate_answer(question: str, context: list[str]) -> str:

    context_text = "\n\n".join(context)

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": (
                    f"Context:\n{context_text}\n\n"
                    f"Question: {question}"
                ),
            },
        ],
    )

    return response.choices[0].message.content.strip()


def ask_rag(question: str, k: int = 2):
    context = retrieve(question, k=k)
    answer = generate_answer(question, context)

    return answer, context

## 15.4 — Create the Golden Evaluation Set

A proper RAG evaluation case should contain:

- `input` — user's question
- `expected_output` — reference answer
- `retrieval_context` — what the retriever actually returned
- `actual_output` — what the generator actually produced

Today every test case contains all four fields because we want to run all five RAG metrics on every case.

In [4]:
GOLDEN_SET = [
    {
        "input": "What is a token in an LLM?",
        "expected_output": (
            "A token is a basic unit of text processed by an LLM. "
            "Both input and output are measured in tokens."
        ),
    },
    {
        "input": "How does RAG reduce hallucination?",
        "expected_output": (
            "RAG retrieves relevant information and provides it to the LLM "
            "as context, which helps ground the answer."
        ),
    },
    {
        "input": "What is an embedding?",
        "expected_output": (
            "An embedding is a vector representation of text that allows "
            "semantically similar texts to be compared."
        ),
    },
]

rag_rows = []

for item in GOLDEN_SET:

    answer, context = ask_rag(item["input"])

    rag_rows.append({
        "input": item["input"],
        "expected_output": item["expected_output"],
        "actual_output": answer,
        "retrieval_context": context,
    })

    print("=" * 70)
    print("Question:", item["input"])
    print("Answer:", answer)

    print("\nRetrieved context:")
    for i, passage in enumerate(context, start=1):
        print(f"{i}. {passage}")

Question: What is a token in an LLM?
Answer: The provided context does not contain information about what a token is in an LLM.

Retrieved context:
1. An embedding is a vector representation of text. Texts with similar meanings have vectors that are close together, which enables semantic search.
2. A hallucination is information generated by an LLM that is factually incorrect or unsupported by its sources.
Question: How does RAG reduce hallucination?
Answer: RAG reduces hallucination by retrieving relevant external documents and providing that information as context to the LLM, grounding its answers in real data.

Retrieved context:
1. RAG grounds an LLM's answer in external documents. The system retrieves relevant information and provides it to the LLM as context, which can reduce hallucinations.
2. Large language models split text into tokens. Tokens are the basic units processed by an LLM, and both input and output are measured in tokens.
Question: What is an embedding?
Answer: An e

## 15.5 — Convert the Pipeline Runs into LLMTestCases

The pipeline has now produced everything DeepEval needs.

```text
Golden Question
      ↓
Retriever
      ↓
retrieval_context
      ↓
Generator
      ↓
actual_output
      ↓
LLMTestCase

In [5]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input=row["input"],
        actual_output=row["actual_output"],
        expected_output=row["expected_output"],
        retrieval_context=row["retrieval_context"],
    )
    for row in rag_rows
]

print("Test cases:", len(test_cases))

Test cases: 3


## 15.6 — Create the Full RAG Metric Suite

We now combine the metrics from Days 12–14.

### Retriever Metrics

- Contextual Precision
- Contextual Recall
- Contextual Relevancy

### Generator Metrics

- Faithfulness
- Answer Relevancy

In [6]:
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
)

metrics = [
    ContextualPrecisionMetric(
        model=judge,
        threshold=0.5,
    ),

    ContextualRecallMetric(
        model=judge,
        threshold=0.5,
    ),

    ContextualRelevancyMetric(
        model=judge,
        threshold=0.5,
    ),

    FaithfulnessMetric(
        model=judge,
        threshold=0.5,
    ),

    AnswerRelevancyMetric(
        model=judge,
        threshold=0.5,
    ),
]

print("Metrics:", len(metrics))

Metrics: 5


## 15.7 — Run the Full Evaluation

Each test case is now evaluated from two perspectives:

```text
                 RAG PIPELINE

Question
   ↓
Retriever ────────────────┐
   ↓                      │
Context                   │
   ↓                      │
Generator                 │
   ↓                      │
Answer                    │
                          │
        EVALUATION        │
                          │
Retriever Metrics ←───────┘
Generator Metrics ← Answer + Context

In [7]:
from deepeval import evaluate

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=async_config,
)

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

c:\Users\T14s\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              What is a token in an LLM?                                                           │
│  │     Actual Output:      The provided context does not contain information about what a token is in an        │
│  │                         LLM.                                                                                 │
│  │     Expected Output:    A token is a basic unit of text processed by an LLM. Both input and output are       │
│  │                         measured in tokens.                                                                  │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Contextual Precision │ 0.00  │ 0.50      │ The score is 0.00 because the first node in           │
│              │                      │       │           │ retrieval contexts discusses embeddings, which is     │
│              │                      │       │           │ unrelated to the input, and the second node           │
│              │                      │       │           │ explains hallucinations, also irrelevant. Since all   │
│              │                      │       │           │ nodes are irrelevant and ranked at the top, the       │
│              │                      │       │           │ score cannot be higher.                               │
│        FAIL  │ Contextual Recall    │ 0.00  │ 0.50      │ The score is 0.00 because none of the sentences in    │
│              │                      │       │           │ the expected output are supported by any              │
│              │                      │       │           │ information in the retrieval context nodes, which     │
│              │                      │       │           │ do not mention or define 'token' or discuss           │
│              │                      │       │           │ input/output measurement in tokens.                   │
│        FAIL  │ Contextual Relevancy │ 0.00  │ 0.50      │ The score is 0.00 because none of the statements in   │
│              │                      │       │           │ the retrieval context address what a token in an      │
│              │                      │       │           │ LLM is; all focus on embeddings, semantic search,     │
│              │                      │       │           │ or hallucinations, which are unrelated to the input   │
│              │                      │       │           │ question.                                             │
│        PASS  │ Faithfulness         │ 1.00  │ 0.50      │ The score is 1.00 because there are no contradi...    │
│        FAIL  │ Answer Relevancy     │ 0.00  │ 0.50      │ The score is 0.00 because the output fails to         │
│              │                      │       │           │ provide any information about what a token is in an   │
│              │                      │       │           │ LLM, making it completely irrelevant to the input     │
│              │                      │       │           

⚠ WARNING: No hyperparameters logged.
» ]8;id=47719;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 10.65s | token cost: 0.011248000000000001 USD)
» Test Results (3 total tests):
   » Pass Rate: 33.33% | Passed: 1 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## 15.8 — Build a Readable RAG Report

A score alone is not enough.

For every case we want to see:

- metric
- score
- PASS / FAIL
- reason

In [8]:
for i, result in enumerate(
    results.test_results,
    start=1
):

    print("\n" + "=" * 75)
    print(f"CASE {i}")
    print("Question:", result.input)
    print("=" * 75)

    for metric_result in result.metrics_data:

        status = (
            "PASS"
            if metric_result.success
            else "FAIL"
        )

        print(
            f"[{status}] "
            f"{metric_result.name:<25} "
            f"score={metric_result.score:.2f}"
        )

        print(
            f"       {metric_result.reason[:200]}"
        )


CASE 1
Question: What is a token in an LLM?
[FAIL] Contextual Precision      score=0.00
       The score is 0.00 because the first node in retrieval contexts discusses embeddings, which is unrelated to the input, and the second node explains hallucinations, also irrelevant. Since all nodes are 
[FAIL] Contextual Recall         score=0.00
       The score is 0.00 because none of the sentences in the expected output are supported by any information in the retrieval context nodes, which do not mention or define 'token' or discuss input/output m
[FAIL] Contextual Relevancy      score=0.00
       The score is 0.00 because none of the statements in the retrieval context address what a token in an LLM is; all focus on embeddings, semantic search, or hallucinations, which are unrelated to the inp
[PASS] Faithfulness              score=1.00
       The score is 1.00 because there are no contradictions; the actual output fully aligns with the retrieval context. Great job maintaining accuracy!
[F

## 15.9 — Diagnose the Pipeline

The goal of evaluation is not simply to collect numbers.

The goal is to answer:

> **What should we fix?**

Use this mental model:

| Metric Failure | Likely Area |
|---|---|
| Contextual Precision | Ranking |
| Contextual Recall | Retrieval coverage |
| Contextual Relevancy | Retrieval noise |
| Faithfulness | Generator grounding |
| Answer Relevancy | Generator usefulness |

A failing answer does not automatically mean the LLM is the problem.

Sometimes the generator received bad context.

In [9]:
for i, result in enumerate(
    results.test_results,
    start=1
):

    print("\n" + "=" * 70)
    print(f"DIAGNOSIS — CASE {i}")
    print(result.input)

    failed_metrics = [
        metric_result.name
        for metric_result in result.metrics_data
        if metric_result.success is False
    ]

    if not failed_metrics:
        print("All metrics passed.")

    else:
        print("Failed metrics:")

        for metric_name in failed_metrics:
            print("-", metric_name)


DIAGNOSIS — CASE 1
What is a token in an LLM?
Failed metrics:
- Contextual Precision
- Contextual Recall
- Contextual Relevancy
- Answer Relevancy

DIAGNOSIS — CASE 2
How does RAG reduce hallucination?
All metrics passed.

DIAGNOSIS — CASE 3
What is an embedding?
Failed metrics:
- Contextual Relevancy


## 15.10 — Read Failures Like an Engineer

Consider a few common patterns.

### Low Recall

```text
Retriever failed to find required information.

# Day 15 — Key Takeaways

Today we built our first **complete RAG evaluation suite**.

We evaluated two different layers:

```text
                 RAG
                  │
        ┌─────────┴─────────┐
        ↓                   ↓
    RETRIEVER            GENERATOR
        │                   │
   Precision            Faithfulness
   Recall               Answer Relevancy
   Relevancy